In [1]:
import ndex2
import json
from ndex2.cx2 import PandasDataFrameToCX2NetworkFactory
from ndex2.nice_cx_network import NiceCXNetwork
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import getpass
from ndex2.client import DecimalEncoder
import csv
import os
import glob

In [2]:
dico_top_pathway_ids = {
    "01": "Autophagy",
    "02": "CellCycle",
    "03": "CellCellCommunication",
    "04": "CellularResponseToStimuli",
    "05": "ChromatinOrganization",
    "06": "CircadianClock",
    "07": "DevelopmentalBiology",
    "08": "DigestionAndAbsorption",
    "09": "Disease",
    "10": "DNARepair",
    "11": "DNAReplication",
    "12": "DrugADME",
    "13": "ExtracellularMatrixOrganization",
    "14": "GeneExpression(Transcription)",
    "15": "Hemostasis",
    "16": "ImmuneSystem",
    "17": "Metabolism",
    "18": "MetabolismOfProteins",
    "19": "MetabolismOfRNA",
    "20": "MuscleContraction",
    "21": "NeuronalSystem",
    "22": "OrganelleBiogenesisAndMaintenance",
    "23": "ProgrammedCellDeath",
    "24": "ProteinLocalization",
    "25": "Reproduction",
    "26": "SensoryPerception",
    "27": "SignalTransduction",
    "28": "TransportOfSmallMolecules",
    "29": "VesicleMediatedTransport"
}

In [ ]:
current_dir = os.getcwd()
REACTOME_WEIGHTED_CLIQUES = glob.glob(os.path.join(current_dir, "../../Results/PathwayComembership/02_WeightedComembershipCliques/*.csv"))

['/home/cbeust/Projects/2026/AbstractionReactomeTopPathways/Scripts/03_PathwayComembershipAbstraction/../../Results/PathwayComembership/02_WeightedComembershipCliques/21_WeightedComembershipClique.csv', '/home/cbeust/Projects/2026/AbstractionReactomeTopPathways/Scripts/03_PathwayComembershipAbstraction/../../Results/PathwayComembership/02_WeightedComembershipCliques/27_WeightedComembershipClique.csv', '/home/cbeust/Projects/2026/AbstractionReactomeTopPathways/Scripts/03_PathwayComembershipAbstraction/../../Results/PathwayComembership/02_WeightedComembershipCliques/05_WeightedComembershipClique.csv', '/home/cbeust/Projects/2026/AbstractionReactomeTopPathways/Scripts/03_PathwayComembershipAbstraction/../../Results/PathwayComembership/02_WeightedComembershipCliques/04_WeightedComembershipClique.csv', '/home/cbeust/Projects/2026/AbstractionReactomeTopPathways/Scripts/03_PathwayComembershipAbstraction/../../Results/PathwayComembership/02_WeightedComembershipCliques/08_WeightedComembershipCl

In [18]:
dico_visual_properties = {
    "default": {
        "network": {
            "NETWORK_BACKGROUND_COLOR": "#FFFFFF"
        },
        "edge": {
            "EDGE_LABEL": "",
            "EDGE_LABEL_COLOR": "#000000",
            "EDGE_LABEL_FONT_FACE": {
                "FONT_FAMILY": "sans-serif",
                "FONT_STYLE": "normal",
                "FONT_WEIGHT": "normal"
            },
            "EDGE_LABEL_FONT_SIZE": 10,
            "EDGE_LINE_STYLE": "solid",
            "EDGE_OPACITY": 0.6666666666666666,
            "EDGE_SOURCE_ARROW_COLOR": "#000000",
            "EDGE_SOURCE_ARROW_SHAPE": "none",
            "EDGE_LINE_COLOR": "#000000",  # Default color (fallback)
            "EDGE_TARGET_ARROW_COLOR": "#000000",
            "EDGE_TARGET_ARROW_SHAPE": "none",
            "EDGE_WIDTH": 2,
        },
        "node": {
            "NODE_BORDER_COLOR": "#333333",
            "NODE_BORDER_STYLE": "solid",
            "NODE_BORDER_WIDTH": 1,
            "NODE_BACKGROUND_COLOR": "#78c7df",
            "NODE_HEIGHT": 20,
            "NODE_LABEL": "",
            "NODE_LABEL_COLOR": "#000000",
            "NODE_LABEL_FONT_FACE": {
                "FONT_FAMILY": "sans-serif",
                "FONT_STYLE": "normal",
                "FONT_WEIGHT": "normal"
            },
            "NODE_LABEL_FONT_SIZE": 12,
            "NODE_LABEL_POSITION": {
                "HORIZONTAL_ALIGN": "center",
                "VERTICAL_ALIGN": "top",
                "HORIZONTAL_ANCHOR": "center",
                "VERTICAL_ANCHOR": "bottom",
                "MARGIN_X": 0,
                "MARGIN_Y": 5,
                "JUSTIFICATION": "center"
            },
            "NODE_SHAPE": "ellipse",
            "NODE_WIDTH": 20,
        }
    },
    # SINGLE edgeMapping with BOTH mappings
    "edgeMapping": {
        "EDGE_LINE_COLOR": {
            "type": "DISCRETE",
            "definition": {
                "attribute": "provenanceScore",
                "map": [
                    {
                        "v": "scoreMICA",
                        "vp": "#4ea75d"  # Green
                    },
                    {
                        "v": "scoreNextStep",
                        "vp": "#ff0000"  # Red (changed from #fd000a for brighter red)
                    }
                ]
            }
        },
        "EDGE_OPACITY": {
            "type": "CONTINUOUS",
            "definition": {
                "attribute": "scoreComembership",
                "lower": 0.0,
                "upper": 1.0,
                "lowerVP": 0.2,
                "upperVP": 1.0
            }
        }
    },
    "nodeMapping": {
        "NODE_LABEL": {
            "type": "PASSTHROUGH",
            "definition": {
                "attribute": "name"
            }
        }
    }
}

In [20]:
counter = 1
client = ndex2.client.Ndex2(username='cecile.beust', password='23BioNet4&These26!')
folder = client.files.create_folder('Reactome96WeightedPathwayComembership', visibility='PUBLIC')
for clique in sorted(REACTOME_WEIGHTED_CLIQUES):
    print(clique)

    comembershipClique = pd.read_csv(clique, sep=",", header=0)
    print(comembershipClique.head())
    comembershipClique.columns = ['entity1', 'entity2', 'scoreComembership', 'provenanceScore']

    pd_factory = PandasDataFrameToCX2NetworkFactory()
    comembershipCliqueCX2 = pd_factory.get_cx2network(comembershipClique, source_field='entity1', target_field='entity2')
    print(comembershipCliqueCX2)
    comembershipCliqueCX2.add_network_attribute('name', f"{dico_top_pathway_ids[f'{counter:02d}']}_WeightedComembershipClique")
    print('network_attributes:', comembershipCliqueCX2.get_network_attributes())
    print('number of nodes:', len(comembershipCliqueCX2.get_nodes()))
    print('number of edges:', len(comembershipCliqueCX2.get_edges()))
    comembershipCliqueCX2.set_visual_properties(dico_visual_properties)
    print(comembershipCliqueCX2.get_nodes())
    net_url = client.save_new_cx2_network(comembershipCliqueCX2.to_cx2(), folder_id=folder, visibility='PUBLIC')
    #res = client.save_new_cx2_network(comembershipCliqueCX2.to_cx2(), visibility='PUBLIC')
    counter += 1


/home/cbeust/Projects/2026/AbstractionReactomeTopPathways/Scripts/03_PathwayComembershipAbstraction/../../Results/PathwayComembership/02_WeightedComembershipCliques/01_WeightedComembershipClique.csv
  entity1 entity2  scoreComembership provenanceScore
0  Q9UGJ0  O43741           1.191886       scoreMICA
1  O60664  Q9UGJ0           1.191886       scoreMICA
2  O60664  Q9Y478           1.191886       scoreMICA
3  O43741  Q9Y478           1.191886       scoreMICA
4  Q9UGJ0  Q9Y478           1.191886       scoreMICA
Name: None
Nodes: 140
Edges: 9730
Node Attributes: 1
Edge Attributes: 3

network_attributes: {'name': 'Autophagy_WeightedComembershipClique'}
number of nodes: 140
number of edges: 9730
{0: {'id': 0, 'v': {'name': 'Q9UGJ0'}, 'x': None, 'y': None, 'z': None}, 1: {'id': 1, 'v': {'name': 'O43741'}, 'x': None, 'y': None, 'z': None}, 2: {'id': 2, 'v': {'name': 'O60664'}, 'x': None, 'y': None, 'z': None}, 3: {'id': 3, 'v': {'name': 'Q9Y478'}, 'x': None, 'y': None, 'z': None}, 4: {'id':